# 04 — Plantes / KNN (supervisé)
**But :** prédire une catégorie botanico-horticole **expertisée**, indépendante des caractéristiques utilisées comme prédicteurs.

Les étiquettes humaines sont indispensables. On ne peut pas prétendre entraîner KNN en déduisant automatiquement la classe de K-Means. Ce notebook utilise une table de labels dédiée (migration V11).


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"database/scripts/plant_quality_report.py").exists())
load_dotenv(ROOT/".env",override=False)
load_dotenv(ROOT/".env.local",override=False)
DSN=os.environ["GARDNAIZE_ML_DATABASE_URL"].replace("postgresql+psycopg://","postgresql://",1)
def get_data():
    with psycopg.connect(DSN) as conn:
        with conn.cursor() as cur:
            cur.execute("""SELECT plant_id,scientific_name,growth_form,exposure,soil_texture,
                water_requirement,hardiness_min_c,height_cm,spread_cm,evergreen,
                quality_status,ready_for_matching
                FROM horticulture.ml_plant_catalog
                WHERE quality_status='VALIDATED'
                  AND outdoor_confirmed IS TRUE
                  AND temperate_confirmed IS TRUE
                ORDER BY plant_id""")
            return pd.DataFrame(cur.fetchall(),columns=[d.name for d in cur.description])
df=get_data()
print("Plantes admissibles :",len(df))
display(df.head())
with psycopg.connect(DSN) as conn:
    with conn.cursor() as cur:
        cur.execute("""SELECT plant_id,label_value FROM horticulture.ml_plant_label
                       WHERE label_type='PLANT_GARDEN_ROLE' AND reviewed=true""")
        labels=pd.DataFrame(cur.fetchall(),columns=["plant_id","label"])
df=df.merge(labels,on="plant_id",how="inner")
print("Plantes labellisées :",len(df))
display(df["label"].value_counts())



In [ ]:
from sklearn.model_selection import StratifiedKFold,cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,classification_report,ConfusionMatrixDisplay
import matplotlib.pyplot as plt
numeric=["hardiness_min_c","height_cm","spread_cm"]
categorical=["growth_form","exposure","soil_texture","water_requirement","evergreen"]
counts=df["label"].value_counts()
assert len(counts)>=2 and counts.min()>=2, "Il faut >=2 catégories et >=2 plantes par catégorie."
assert len(df)>=10, "Jeu de données trop petit pour une évaluation utile."
X=df[numeric+categorical];y=df["label"]
prep=ColumnTransformer([
 ("num",Pipeline([("impute",SimpleImputer(strategy="median",keep_empty_features=True)),("scale",StandardScaler())]),numeric),
 ("cat",Pipeline([("impute",SimpleImputer(strategy="constant",fill_value="UNKNOWN")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical)
])
cv=StratifiedKFold(n_splits=min(3,int(counts.min())),shuffle=True,random_state=42)
knn=Pipeline([("preprocess",prep),("classifier",KNeighborsClassifier(n_neighbors=3))])
baseline=DummyClassifier(strategy="most_frequent")
for name,estimator in [("KNN",knn),("Baseline",baseline)]:
 pred=cross_val_predict(estimator,X,y,cv=cv)
 print(name,"accuracy",round(accuracy_score(y,pred),3),"balanced",round(balanced_accuracy_score(y,pred),3),"macro F1",round(f1_score(y,pred,average="macro",zero_division=0),3))
 if name=="KNN":
  print(classification_report(y,pred,zero_division=0))
  ConfusionMatrixDisplay.from_predictions(y,pred,labels=sorted(counts.index),xticks_rotation=35)
  plt.tight_layout();plt.show()

